In [5]:
import pandas as pd

In [23]:
def read_fasta(path):
    d = {}
    with open(path) as f:
        key = None
        for line in f:
            line = line.strip()
            if line.startswith(">"):
                key = line[1:]
                d[key] = ""
            else:
                d[key] += line
    return d

def dict_to_fasta(seq_dict, fasta_path):
    """
    seq_dict: {name: seq}
    fasta_path: 输出 fasta 文件路径
    """
    with open(fasta_path, "w") as f:
        for name, seq in seq_dict.items():
            f.write(f">{name}\n")
            # 可选：按 60 列换行，方便阅读
            for i in range(0, len(seq), 60):
                f.write(seq[i:i+60] + "\n")
                
def make_dms(seq, start=None, end=None, aa_list="ACDEFGHIKLMNPQRSTVWY"):
    n = len(seq)
    start = 1 if start is None else start
    end = n if end is None else end

    mut_dict = {}

    for pos in range(start, end + 1):  # 1-based
        wt = seq[pos - 1]
        for aa in aa_list:
            if aa == wt:
                continue
            mut_seq = seq[:pos - 1] + aa + seq[pos:]
            key = f"{wt}{pos}{aa}"   # 例如 A205K
            mut_dict[key] = mut_seq

    return mut_dict

def make_dms_natural(ref_seq, allowed_aas, start=None, end=None):
    """
    ref_seq: 参考氨基酸序列字符串
    allowed_aas: dict, position(1-based) -> list of allowed amino acids (自然界出现过的)
    start, end: 1-based 位点区间（包含）
    """
    mutants = {}
    L = len(ref_seq)

    if start is None:
        start = 1
    if end is None:
        end = L

    # 遍历位点（1-based）
    for pos in range(start, end + 1):
        # 如果这个位点不在 allowed_aas 里，直接跳过
        if pos not in allowed_aas:
            continue

        wt = ref_seq[pos - 1]  # 序列是 0-based

        # 只在“出现过的 aa”里做突变
        for aa in allowed_aas[pos]:
            if aa == wt:
                # 不做 WT->WT 的“假突变”
                continue

            seq_list = list(ref_seq)
            seq_list[pos - 1] = aa
            mut_seq = ''.join(seq_list)

            mut_name = f"{wt}{pos}{aa}"  # 比如 A190K
            mutants[mut_name] = mut_seq

    return mutants


In [7]:
vaccine_dict = read_fasta('./Sequence/Vaccine_strain.fasta')
Crick_early = pd.read_csv('../../data/data_40/all.csv')
Crick_41 = pd.read_csv('../../data/data_40/Crick_41.csv')
Crick_42 = pd.read_csv('../../data/data_40/Crick_42.csv')
Crick_43 = pd.read_csv('../../data/data_40/Crick_43.csv')

In [8]:
Crick_all = pd.concat([Crick_early, Crick_41, Crick_42, Crick_43])
Crick_filt = Crick_all[(Crick_all['seq_c'].str.len() == 566) & (Crick_all['Type'] == 'H3N2')].reset_index(drop=True).copy()

In [20]:
#只取 17–345 位（Python 索引 16:345），展开成 DataFrame
expanded_df = Crick_filt['seq_c'].drop_duplicates().apply(lambda x: pd.Series(list(x)))
expanded_df.columns = range(1, 567)

# 为每个位点收集出现过的氨基酸（去掉缺失）
allowed_aas = {pos: sorted(expanded_df[pos].dropna().unique().tolist()) for pos in expanded_df.columns}

In [21]:
ref_seq = vaccine_dict['A/Croatia/10136RV/2023|HA']

dms_17_345 = make_dms_natural(ref_seq,allowed_aas=allowed_aas,start=17,end=345)
print(len(dms_17_345))

759


In [24]:
dict_to_fasta(dms_17_345, "./Sequence/DMS_17_345(nature).fasta")
